In [43]:
import os
import shutil
import random
import math

In [ ]:
SINGAN_SYNTHETIC_SAMPLES = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Output/RandomSamples'
# TODO: Add diffusion-generated random samples

# Original data
TRAINING_DATA_FOLDER = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_2/augmented_dataset_expansion_factor_0'
ALL_TRAINING_IMAGES = '/home/miguel/GI/0 - Data Exploration & Analysis/GI-Roberta/gi-roberta-dataset/full_dataset'

PER_SUBJECT = '/home/miguel/GI/1.5 - Synthetic Data Generation/experiments/synthetic-data-training-benchmark/singan/data/per-subject'

In [47]:
# Prepare the per-subject dataset

In [88]:
previously_selected_synthetic_images = {} # filename: previously selected synthetic images
metadata = {}
for cv_subject in ['FD-027', 'FD-029', 'FD-030', 'FD-031', 'FD-032']:
    image_paths = [os.path.join(ALL_TRAINING_IMAGES, f) for f in os.listdir(ALL_TRAINING_IMAGES) if f.startswith(cv_subject) and 'image' in f]
    image_paths = sorted(image_paths)

    cv_subject_data_path = os.path.join(PER_SUBJECT, cv_subject)
    shutil.rmtree(cv_subject_data_path, ignore_errors=True)
    os.makedirs(cv_subject_data_path, exist_ok=True)

    for i in range(0, len(image_paths) - 16, 8):
        block_folder = os.path.join(cv_subject_data_path, f'block_{i // 8}_{i}_to_{i + 24}')
        os.makedirs(block_folder, exist_ok=True)

        eval_images = image_paths[i:i + 24]
        training_images = image_paths[:max(0, i - 2)] + image_paths[min(i+24, len(image_paths)-1):]

        train_real_data_path = os.path.join(block_folder, 'train_real')
        train_synthetic_data_path = os.path.join(block_folder, 'train_synthetic')
        eval_data_path = os.path.join(block_folder, 'eval')
        os.makedirs(train_real_data_path, exist_ok=True)
        os.makedirs(train_synthetic_data_path, exist_ok=True)
        os.makedirs(eval_data_path, exist_ok=True)        

        for img in training_images:
            if 'synthetic' not in img:
                shutil.copy(img, train_real_data_path)
                mask_path = img.replace('image', 'mask')
                shutil.copy(mask_path, train_real_data_path)

        for img in eval_images:
            shutil.copy(img, eval_data_path)
            mask_path = img.replace('image', 'mask')
            shutil.copy(mask_path, eval_data_path)

        for j in range(5):
            train_synthetic_subfolder = os.path.join(train_synthetic_data_path, f'subfolder_{j}')
            os.makedirs(train_synthetic_subfolder, exist_ok=True)
            for img in training_images:
                if 'synthetic' in img:
                    continue
                synthetic_data_folder_name = img.split('/')[-1][:-4]
                synthetic_data_folder = os.path.join(SINGAN_SYNTHETIC_SAMPLES, synthetic_data_folder_name, 'gen_start_scale=0')
                # Randomly select an image not in previously_selected_synthetic_images[img]

                synthetic_imgs = [f for f in os.listdir(synthetic_data_folder) if 'img' in f]
                already_used = set(previously_selected_synthetic_images.get(img, []))
                available = [f for f in synthetic_imgs if f not in already_used]

                # Randomly select the desired number (or all if fewer are available)
                selected = random.sample(available, 1)

                new_path = os.path.join(train_synthetic_subfolder, os.path.basename(img).replace('.png', f'_synthetic_{selected[0].replace('_img.png', '')}.png'))
                image_path = os.path.join(synthetic_data_folder, selected[0])
                shutil.copy(image_path, new_path)

                mask_path = img.replace('image', 'mask')
                new_mask_path = new_path.replace('image', 'mask')
                shutil.copy(mask_path, new_mask_path)

In [92]:
CROSS_SUBJECT = '/home/miguel/GI/1.5 - Synthetic Data Generation/experiments/synthetic-data-training-benchmark/singan/data/cross-subject'

In [93]:
import os
import random
import shutil

# Reproducibility (optional)
random.seed(42)

previously_selected_synthetic_images = {}  # {real_image_path: set([synthetic_filenames])}
metadata = {}

# Helper to ensure a fresh dir
def remake_dir(p):
    shutil.rmtree(p, ignore_errors=True)
    os.makedirs(p, exist_ok=True)

# Collect all real image paths once
all_real_images = sorted(
    os.path.join(ALL_TRAINING_IMAGES, f)
    for f in os.listdir(ALL_TRAINING_IMAGES)
    if f.endswith(('.png', '.jpg', '.jpeg', '.tif')) and ('image' in f)
)

# Utility: copy image+mask pair
def copy_pair(img_path, dest_dir):
    shutil.copy(img_path, dest_dir)
    mask_path = img_path.replace('image', 'mask')
    shutil.copy(mask_path, dest_dir)

# Main LOO loop: each cv_subject is the held-out subject for EVAL
for cv_subject in ['FD-027', 'FD-029', 'FD-030', 'FD-031', 'FD-032']:
    # Split: eval = this subject; train = all other subjects
    eval_images = [p for p in all_real_images if os.path.basename(p).startswith(cv_subject)]
    training_images = [p for p in all_real_images if not os.path.basename(p).startswith(cv_subject)]

    # Output dirs
    cv_subject_root = os.path.join(CROSS_SUBJECT, f'loo_{cv_subject}')
    remake_dir(cv_subject_root)

    train_real_data_path = os.path.join(cv_subject_root, 'train_real')
    train_synthetic_data_path = os.path.join(cv_subject_root, 'train_synthetic')
    eval_data_path = os.path.join(cv_subject_root, 'eval')
    os.makedirs(train_real_data_path, exist_ok=True)
    os.makedirs(train_synthetic_data_path, exist_ok=True)
    os.makedirs(eval_data_path, exist_ok=True)

    # ---- Copy EVAL (held-out subject) ----
    for img in eval_images:
        copy_pair(img, eval_data_path)

    # ---- Copy TRAIN_REAL (all other subjects) ----
    for img in training_images:
        copy_pair(img, train_real_data_path)

    # ---- Build 5 synthetic datasets (one synthetic per real image per dataset) ----
    for j in range(5):
        synth_subset_dir = os.path.join(train_synthetic_data_path, f'subfolder_{j}')
        os.makedirs(synth_subset_dir, exist_ok=True)

        for real_img in training_images:
            # Find the folder containing the 50 synthetics for this real image
            synthetic_data_folder_name = os.path.splitext(os.path.basename(real_img))[0]
            synthetic_data_folder = os.path.join(
                SINGAN_SYNTHETIC_SAMPLES, synthetic_data_folder_name, 'gen_start_scale=0'
            )
            if not os.path.isdir(synthetic_data_folder):
                # Skip gracefully if missing
                # (You might log this if you expect all to exist)
                continue

            # List candidate synthetic images (e.g., files containing 'img')
            synthetic_imgs = sorted(f for f in os.listdir(synthetic_data_folder) if 'img' in f)

            # Track & avoid re-using the exact same synthetic across subfolders until exhausted
            used = previously_selected_synthetic_images.get(real_img, set())
            available = [f for f in synthetic_imgs if f not in used]

            if not available:
                # All have been used at least once; reset to allow reuse (or skip if you prefer)
                used = set()
                available = synthetic_imgs

            # Pick ONE synthetic for this real image for this subset
            selected = random.sample(available, 1)[0]
            used.add(selected)
            previously_selected_synthetic_images[real_img] = used

            # Copy the selected synthetic image and pair it with the real mask
            # Name it to encode which synthetic was picked (strip trailing "_img.png" if present)
            stem = selected.replace('_img.png', '').replace('.png', '')
            new_img_name = os.path.basename(real_img).replace('.png', f'_synthetic_{stem}.png')
            new_img_path = os.path.join(synth_subset_dir, new_img_name)

            shutil.copy(os.path.join(synthetic_data_folder, selected), new_img_path)

            # Synthetic mask = source real mask (since synthetic was derived from that real)
            src_mask_path = real_img.replace('image', 'mask')
            new_mask_path = new_img_path.replace('image', 'mask')
            shutil.copy(src_mask_path, new_mask_path)

In [94]:
new_mask_path

'/home/miguel/GI/1.5 - Synthetic Data Generation/experiments/synthetic-data-training-benchmark/singan/data/cross-subject/loo_FD-032/train_synthetic/subfolder_4/FD-031-slice-72-mask_synthetic_7.png'